# NautilusTrader: the one line that makes its fills realistic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarptandoven/queuetruth/blob/main/examples/notebooks/05_nautilustrader.ipynb)

NautilusTrader's default `FillModel` fills a resting limit order as soon as the market touches its price, as if
you were first in the queue. This notebook gets a better setting two ways, the atlas when you have no order-level
data and a fit when you do, applies it with one line, and then checks a NautilusTrader backtest order by order:

```python
engine.add_venue(..., **qtn.venue_options(qt.fit("nautilus", data="my-dataset")))
engine.add_venue(..., **qtn.venue_options(qt.atlas.calibration("nautilus", tier="T1")))
```

Data: queuetruth's **synthetic** dataset and the repository's **synthetic** atlas (invented numbers). The numbers
only show the workflow; they say nothing about a real venue. Every result reads the same way: `.summary` (the
answer), `.table()`, `.to_html()`, `.save(path)`, and a compact view in a notebook (`display(result)`).

In [1]:
# Setup: on Colab (or anywhere queuetruth is missing) this installs it from PyPI; elsewhere it does nothing.
import importlib
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("queuetruth") is None:
    try:
        from IPython import get_ipython

        shell = get_ipython()
    except ImportError:
        shell = None
    if shell is not None:  # %pip lets Colab offer a restart when an already-imported package was upgraded
        shell.run_line_magic("pip", "install -q queuetruth pandas matplotlib")
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "queuetruth", "pandas", "matplotlib"])
    importlib.invalidate_caches()
if importlib.util.find_spec("queuetruth") is None or importlib.util.find_spec("queuetruth.frames") is None:
    raise RuntimeError(
        "this notebook needs queuetruth.frames (first released after queuetruth 0.1.1): pip install -U queuetruth"
    )

In [2]:
import importlib.util
from pathlib import Path

import pandas as pd
from IPython.display import display

import queuetruth as qt

pd.set_option("display.width", 140)
WORK = Path("qt-notebooks") / "05_nautilustrader"
WORK.mkdir(parents=True, exist_ok=True)

# queuetruth's synthetic order-level dataset: two futures (QTAU6, QTBU6), two days, invented prices.
DATA = WORK / "synth"
if not (DATA / "manifest.json").exists():
    qt.fixtures_synth(str(DATA))

# The repository's synthetic atlas (invented numbers, no market data): the checkout's copy, else the one on GitHub.
# Leave `source=` out to use the published atlas (it asks before downloading anything).
_local = Path(qt.__file__).resolve().parents[2] / "tests" / "fixtures" / "atlas_synth"
ATLAS = (
    str(_local)
    if (_local / "atlas" / "v1" / "manifest.json").exists()
    else "https://raw.githubusercontent.com/sarptandoven/queuetruth/main/tests/fixtures/atlas_synth/atlas/v1/"
)

# Twelve resting limit orders on the synthetic futures, as a bar backtest reports them: placed at `placed` (UTC),
# and filled 5 s later when `filled` (the touch rule), else cancelled 30 s later.
ORDERS = pd.DataFrame(
    [
        ("O-1", "QTBU6", "sell", 1, 5008.50, "2026-06-16 14:38:42", True),
        ("O-2", "QTAU6", "buy", 2, 4999.75, "2026-06-16 14:38:05", False),
        ("O-3", "QTBU6", "sell", 3, 5010.25, "2026-06-15 17:09:27", True),
        ("O-4", "QTAU6", "buy", 1, 4999.50, "2026-06-16 17:32:15", False),
        ("O-5", "QTAU6", "sell", 2, 5002.75, "2026-06-15 15:39:13", True),
        ("O-6", "QTAU6", "buy", 3, 5004.00, "2026-06-15 19:04:05", False),
        ("O-7", "QTBU6", "buy", 1, 5008.75, "2026-06-15 15:36:19", True),
        ("O-8", "QTBU6", "sell", 2, 5008.50, "2026-06-16 16:58:59", False),
        ("O-9", "QTAU6", "buy", 3, 4999.50, "2026-06-16 14:39:20", True),
        ("O-10", "QTAU6", "buy", 1, 4999.50, "2026-06-16 17:25:05", False),
        ("O-11", "QTBU6", "buy", 2, 5008.25, "2026-06-16 16:12:29", True),
        ("O-12", "QTAU6", "buy", 3, 5001.25, "2026-06-15 15:05:05", False),
    ],
    columns=["order", "symbol", "side", "qty", "price", "placed", "filled"],
)
ORDERS["placed"] = pd.to_datetime(ORDERS["placed"], utc=True)
ORDERS["done"] = ORDERS["placed"] + pd.to_timedelta(ORDERS["filled"].map({True: 5, False: 30}), unit="s")


def have(*modules, hint=None):
    """True when every module is importable; else prints which are missing and how to get them."""
    missing = [m for m in modules if importlib.util.find_spec(m) is None]
    if missing:
        print(f"skipped: {', '.join(missing)} not installed here ({hint or 'pip install ' + ' '.join(missing)})")
    return not missing


print("queuetruth", qt.__version__, "| SYNTHETIC data and a SYNTHETIC atlas: the numbers show the workflow only")

queuetruth 0.3.0 | SYNTHETIC data and a SYNTHETIC atlas: the numbers show the workflow only


## 1. No data yet: the atlas's closest setting

`qt.atlas.calibration` reads the atlas's closest NautilusTrader setting for a liquidity tier, as the same kind of
result `qt.fit` returns. Without `source=` it reads the published atlas, after asking.

In [3]:
cal = qt.atlas.calibration("nautilus", tier="T1", source=ATLAS, yes=True)
display(cal)

setting,tier,horizon,bias_pp,ci_lo_pp,ci_hi_pp,verdict,days,model_id
"NautilusTrader queue position, touch probability 0.5",T1_top100,1min,-0.6,-0.9,-0.3,too_pessimistic,3,nt_queue_l2_prob_touch[mode=faithful;p=0.5]


In [4]:
display(cal.settings)

{'prob_fill_on_limit': 0.5,
 'queue_position': True,
 'book_type': 'L2_MBP',
 'trade_execution': True}

## 2. Your data: fit `prob_fill_on_limit`

`qt.fit` fits the setting on the first 70% of the data and compares it with the library default on the rest. The
snippet is written only when the fit beats the default there.

In [5]:
fit = qt.fit(backtester="nautilus", data=DATA, split="70%", out=WORK / "fit")
display(fit)

horizon,fitted_bias_pp,default_bias_pp,fitted_logloss,default_logloss,result
100ms,0.208833,0.4,0.00295426,0.055263,better
1s,1.67066,3.2,0.0236271,0.442097,better
10s,6.4064,14.446,0.136099,1.99342,better
1min,2.80865,9.2269,0.135294,1.27358,better
5min,-0.546193,2.17241,0.0433468,0.308258,better


In [6]:
display(fit.table())

,horizon,fitted_bias_pp,default_bias_pp,fitted_logloss,default_logloss,result
0,100ms,0.208833,0.400000,0.002954,0.055263,better
1,1s,1.670662,3.200000,0.023627,0.442097,better
2,10s,6.406399,14.445987,0.136099,1.993421,better
3,1min,2.808653,9.226897,0.135294,1.273575,better
4,5min,-0.546193,2.172405,0.043347,0.308258,better


## 3. The one line

`queuetruth_nautilus.venue_options` turns the result into `add_venue` keywords: the fill model with the fitted
`prob_fill_on_limit`, `book_type=L2_MBP`, and `queue_position` when the fit is queue-gated. Without NautilusTrader
installed the cell says so and shows the settings the line would apply.

In [7]:
if have("nautilus_trader", "queuetruth_nautilus"):
    import queuetruth_nautilus as qtn

    try:  # nautilus_trader 2.x
        from nautilus_trader.backtest import BacktestEngine, BacktestEngineConfig
        from nautilus_trader.model import AccountType, Currency, Money, OmsType, Venue
    except ImportError:  # 1.231.x
        from nautilus_trader.backtest.engine import BacktestEngine, BacktestEngineConfig
        from nautilus_trader.model.enums import AccountType, OmsType
        from nautilus_trader.model.identifiers import Venue
        from nautilus_trader.model.objects import Currency, Money

    USD = Currency.from_str("USD")
    engine = BacktestEngine(config=BacktestEngineConfig())
    engine.add_venue(
        venue=Venue("XCME"),
        oms_type=OmsType.NETTING,
        account_type=AccountType.MARGIN,
        base_currency=USD,
        starting_balances=[Money(1_000_000, USD)],
        **qtn.venue_options(fit, random_seed=42),
    )
    print("venue added with", sorted(qtn.venue_options(fit, random_seed=42)))
else:
    print("the settings the line applies:", fit.settings)

skipped: nautilus_trader, queuetruth_nautilus not installed here (pip install nautilus_trader queuetruth_nautilus)
the settings the line applies: {'prob_fill_on_limit': 0.5220820189274448, 'queue_position': False, 'book_type': 'L2_MBP', 'trade_execution': True}


## 4. How wrong was the backtest?

`qt.check` replays a NautilusTrader backtest's own resting limit orders in the exact queue of the data. Here the
backtest is twelve orders written as `generate_orders_report()` writes them; use your own report.

In [8]:
report = WORK / "orders_report.csv"
rows = [
    {
        "client_order_id": o.order,
        "venue_order_id": o.order.replace("O-", "V-"),
        "instrument_id": f"{o.symbol}.XCME",
        "side": o.side.upper(),
        "type": "LIMIT",
        "quantity": o.qty,
        "price": f"{o.price:.2f}",
        "time_in_force": "GTC",
        "post_only": False,
        "reduce_only": False,
        "status": "FILLED" if o.filled else "CANCELED",
        "filled_qty": o.qty if o.filled else 0,
        "avg_px": f"{o.price:.2f}" if o.filled else "",
        "ts_init": o.placed.value,
        "ts_last": o.done.value,
    }
    for o in ORDERS.itertuples()
]
pd.DataFrame(rows).to_csv(report, index=False)
res = qt.check(report, data=DATA, horizon="1s", window="10s", out=WORK / "check")
display(res)

order,symbol,side,price,qty,entry,entry_local,backtest_filled,backtest_fill_after_s,real_filled,real_fill_after_s,queue_ahead,outcome,pnl_gap
sim:nautilus:backtest:O-1,QTBU6,sell,5008.5,1,2026-06-16T14:38:42.000+00:00,2026-06-16 14:38:42.000 UTC,True,5,True,2.497,1,both,0
sim:nautilus:backtest:O-10,QTAU6,buy,4999.5,1,2026-06-16T17:25:05.000+00:00,2026-06-16 17:25:05.000 UTC,False,,False,,12,neither,0
sim:nautilus:backtest:O-11,QTBU6,buy,5008.25,2,2026-06-16T16:12:29.000+00:00,2026-06-16 16:12:29.000 UTC,True,5,False,,0,phantom,25
sim:nautilus:backtest:O-12,QTAU6,buy,5001.25,3,2026-06-15T15:05:05.000+00:00,2026-06-15 15:05:05.000 UTC,False,,True,0.901,17,real_only,0
sim:nautilus:backtest:O-2,QTAU6,buy,4999.75,2,2026-06-16T14:38:05.000+00:00,2026-06-16 14:38:05.000 UTC,False,,False,,8,neither,0
sim:nautilus:backtest:O-3,QTBU6,sell,5010.25,3,2026-06-15T17:09:27.000+00:00,2026-06-15 17:09:27.000 UTC,True,5,False,,49,phantom,37.5
sim:nautilus:backtest:O-4,QTAU6,buy,4999.5,1,2026-06-16T17:32:15.000+00:00,2026-06-16 17:32:15.000 UTC,False,,True,0.371,1,real_only,6.25
sim:nautilus:backtest:O-5,QTAU6,sell,5002.75,2,2026-06-15T15:39:13.000+00:00,2026-06-15 15:39:13.000 UTC,True,5,False,,52,phantom,25
sim:nautilus:backtest:O-6,QTAU6,buy,5004,3,2026-06-15T19:04:05.000+00:00,2026-06-15 19:04:05.000 UTC,False,,False,,5,neither,0
sim:nautilus:backtest:O-7,QTBU6,buy,5008.75,1,2026-06-15T15:36:19.000+00:00,2026-06-15 15:36:19.000 UTC,True,5,False,,22,phantom,6.25


In [9]:
display(res.table()[["order", "symbol", "side", "price", "entry_local", "backtest_filled", "real_filled", "outcome"]])

,order,symbol,side,price,entry_local,backtest_filled,real_filled,outcome
0,sim:nautilus:backtest:O-1,QTBU6,sell,5008.5,2026-06-16 14:38:42.000 UTC,True,True,both
1,sim:nautilus:backtest:O-10,QTAU6,buy,4999.5,2026-06-16 17:25:05.000 UTC,False,False,neither
2,sim:nautilus:backtest:O-11,QTBU6,buy,5008.25,2026-06-16 16:12:29.000 UTC,True,False,phantom
3,sim:nautilus:backtest:O-12,QTAU6,buy,5001.25,2026-06-15 15:05:05.000 UTC,False,True,real_only
4,sim:nautilus:backtest:O-2,QTAU6,buy,4999.75,2026-06-16 14:38:05.000 UTC,False,False,neither
5,sim:nautilus:backtest:O-3,QTBU6,sell,5010.25,2026-06-15 17:09:27.000 UTC,True,False,phantom
6,sim:nautilus:backtest:O-4,QTAU6,buy,4999.5,2026-06-16 17:32:15.000 UTC,False,True,real_only
7,sim:nautilus:backtest:O-5,QTAU6,sell,5002.75,2026-06-15 15:39:13.000 UTC,True,False,phantom
8,sim:nautilus:backtest:O-6,QTAU6,buy,5004,2026-06-15 19:04:05.000 UTC,False,False,neither
9,sim:nautilus:backtest:O-7,QTBU6,buy,5008.75,2026-06-15 15:36:19.000 UTC,True,False,phantom


The same answer as a page to share, and as the JSON document:

In [10]:
print(res.save(WORK / "check.html"))
print(res.save(WORK / "check.json"))

qt-notebooks/05_nautilustrader/check.html
qt-notebooks/05_nautilustrader/check.json
